# Inspiring set
Real poems by the 4 showcase poets that show their form most clearly. The generator doesn't use these,
they're what we compare the generated poems against (Ritchie: is it typical? is it new?).

Picked by rule, no hand-picking: the poet's rhyme scheme, the poet's line length, 12-400 lines long
(long ones are cut to their first 40 lines when printed).
Once the trained model is back, the generated poems go at the bottom of this notebook to compare.

In [3]:
%%capture
%run rhyme.ipynb

In [4]:
import numpy as np

profiles = pd.read_csv(DATA + "author_profiles.csv", index_col=0)
showcase = profiles[profiles["unknown_ends"] <= 0.1].sort_values("lines", ascending=False).groupby("scheme").head(1)
print(showcase[["scheme", "syl_min", "syl_max", "lines"]])

poems = pd.read_parquet(DATA + "pdp_clean.parquet")
poems = poems[poems["author"].isin(showcase.index)]
print(poems["author"].value_counts())

                           scheme  syl_min  syl_max  lines
poet                                                      
Henry Wadsworth Longfellow   none        8       10  54938
Madison Julius Cawein        ABCB        7       10  50167
Algernon Charles Swinburne   AABB        8       11  42005
George Gordon Byron          ABAB       10       11  33216
author
Madison Julius Cawein         1253
Algernon Charles Swinburne     459
Henry Wadsworth Longfellow     295
George Gordon Byron            187
Name: count, dtype: int64


## Score every poem
- `syl`: median syllables per line
- `fit`: share of the poem's 4-line blocks that have the poet's scheme (cut into blocks the same way as in the
  author profiles). For Longfellow ("none") it's the share of lines that don't
  rhyme with any line close by (so sonnets, which rhyme ABBA, don't sneak in).

In [5]:
def rhymes(a, b):
    return rhyme(a, b) >= 0.9


def label(block):
    a, b, c, d = block
    if rhymes(a, b) and rhymes(c, d):
        return "AABB"
    if rhymes(a, c) and rhymes(b, d):
        return "ABAB"
    if rhymes(b, d):
        return "ABCB"
    return "none"


def best_labels(ends):
    # the starting point with the most rhymed blocks, same as the author profiles
    best = None
    for start in range(4):
        labels = [label(ends[i:i + 4]) for i in range(start, len(ends) - 3, 4)]
        if best is None or sum(x != "none" for x in labels) > sum(x != "none" for x in best):
            best = labels
    return best


def fit(ends, scheme):
    if scheme == "none":
        # share of lines that don't rhyme with any of the 3 lines before or after (also catches ABBA sonnets)
        near = [ends[max(i - 3, 0):i] + ends[i + 1:i + 4] for i in range(len(ends))]
        return 1 - np.mean([any(rhymes(e, w) for w in n) for e, n in zip(ends, near)])
    return np.mean([x == scheme for x in best_labels(ends)])  # same blocks as the author profiles


rows = []
for poet, title, text in zip(poems["author"], poems["title"], poems["text"]):
    lines = [words_of(line) for line in text.split("\n")]
    lines = [w for w in lines if w]
    if not 12 <= len(lines) <= 400:  # at least 3 blocks of 4, and not a whole book
        continue
    scheme = showcase.loc[poet, "scheme"]
    rows.append({"poet": poet, "title": title, "text": text, "n_lines": len(lines),
                 "syl": int(np.median([sum(syllables(w) for w in words) for words in lines])),
                 "fit": round(fit([words[-1] for words in lines], scheme), 2)})
scored = pd.DataFrame(rows)
opening = scored["text"].map(lambda t: " ".join(words_of(t)[:8]))
scored = scored[~opening.duplicated()]  # same poem under 2 titles

# keep the poet's line length
lo = scored["poet"].map(showcase["syl_min"])
hi = scored["poet"].map(showcase["syl_max"])
scored = scored[(scored["syl"] >= lo) & (scored["syl"] <= hi)]
print(scored["poet"].value_counts())

poet
Madison Julius Cawein         947
Algernon Charles Swinburne    274
Henry Wadsworth Longfellow    199
George Gordon Byron            50
Name: count, dtype: int64


## The set
The 8 best per poet by `fit`, shorter poems first when it's a tie. A poem has to fit at least half the time
(80% for Longfellow, since half the lines of an ABCB poem don't rhyme either), so a poet can end up with
fewer than 8 (Byron: most of his poems here are early ones in other forms).

In [6]:
need = np.where(scored["poet"].map(showcase["scheme"]) == "none", 0.8, 0.5)
scored = scored[scored["fit"] >= need]
inspiring = scored.sort_values(["fit", "n_lines"], ascending=[False, True]).groupby("poet").head(8)
inspiring = inspiring.sort_values(["poet", "fit"], ascending=[True, False])
print(inspiring[["poet", "title", "n_lines", "syl", "fit"]].to_string(index=False))

                      poet                                                                                               title  n_lines  syl  fit
Algernon Charles Swinburne                                                                                          Stage Love       16   11 1.00
Algernon Charles Swinburne                                                                  Prologue to The Revenger's Tragedy       36   10 1.00
Algernon Charles Swinburne                                                                    Prologue to The Duchess of Malfy       36   10 1.00
Algernon Charles Swinburne                                                                       Prologue to The Spanish Gipsy       40   10 1.00
Algernon Charles Swinburne                                                                          Prologue to Doctor Faustus       48   11 0.92
Algernon Charles Swinburne                                                                                             Eroti

## The poems

In [7]:
for poet, group in inspiring.groupby("poet"):
    print("=" * 60)
    print(poet.upper(), "-", showcase.loc[poet, "scheme"], f"{showcase.loc[poet, 'syl_min']}-{showcase.loc[poet, 'syl_max']} syllables")
    print("=" * 60)
    for title, text, fit_, n in zip(group["title"], group["text"], group["fit"], group["n_lines"]):
        print(f"\n## {title}  ({n} lines, fit {fit_})\n")
        lines = text.strip().split("\n")
        print("\n".join(lines[:40]))
        if len(lines) > 40:
            print(f"... ({len(lines) - 40} more lines)")
    print()

ALGERNON CHARLES SWINBURNE - AABB 8-11 syllables

## Stage Love  (16 lines, fit 1.0)

When the game began between them for a jest,
He played king and she played queen to match the best;
Laughter soft as tears, and tears that turned to laughter,
These were things she sought for years and sorrowed after.
Pleasure with dry lips, and pain that walks by night;
All the sting and all the stain of long delight;
These were things she knew not of, that knew not of her,
When she played at half a love with half a lover.
Time was chorus, gave them cues to laugh or cry;
They would kill, befool, amuse him, let him die;
Set him webs to weave to-day and break to-morrow,
Till he died for good in play, and rose in sorrow.
What the years mean; how time dies and is not slain;
How love grows and laughs and cries and wanes again;
These were things she came to know, and take their measure,
When the play was played out so for one man's pleasure.

## Prologue to The Revenger's Tragedy  (36 lines, fit 1.0)

Fire

## Generated poems
The final pool from `evaluate.ipynb` (5 poems per poet for each system and λ) next to the poems above:
- line length: median syllables per line
- form: the same `fit` as above (share of blocks in the poet's scheme, or of unrhymed lines for Longfellow)
- copying: lines with 5 words in a row from one of that poet's inspiring poems

In [8]:
import json

final = json.load(open("../outputs/final_poems.json", encoding="utf-8"))


def stats(text, scheme):
    lines = [w for w in (words_of(line) for line in text.split("\n")) if w]
    return np.median([sum(syllables(w) for w in ws) for ws in lines]), fit([ws[-1] for ws in lines], scheme)


def grams(words, n=5):
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


rows = []
for poet in showcase.index:
    scheme = showcase.loc[poet, "scheme"]
    real = inspiring[inspiring["poet"] == poet]
    real_grams = set()
    for text in real["text"]:
        for line in text.split("\n"):
            real_grams |= grams(words_of(line))
    rows.append({"poet": poet, "who": "real", "syl": real["syl"].median(), "fit": round(real["fit"].mean(), 2), "copied": ""})
    for system in ["B1", "M"]:
        for lam in sorted({r["lam"] for r in final if r["poet"] == poet}):
            gen = [r["poem"] for r in final if r["poet"] == poet and r["system"] == system and r["lam"] == lam]
            gen_stats = [stats(p, scheme) for p in gen]
            copied = np.mean([bool(grams(words_of(line)) & real_grams) for p in gen for line in p.split("\n")])
            rows.append({"poet": poet, "who": f"{system} λ {lam}", "syl": np.median([s for s, _ in gen_stats]),
                         "fit": round(np.mean([f for _, f in gen_stats]), 2), "copied": f"{copied:.0%}"})
print(pd.DataFrame(rows).to_string(index=False))

                      poet       who  syl  fit copied
Henry Wadsworth Longfellow      real  8.0 0.99       
Henry Wadsworth Longfellow  B1 λ 1.0  8.0 0.94     0%
Henry Wadsworth Longfellow   M λ 1.0  8.0 0.87     0%
     Madison Julius Cawein      real  8.0 1.00       
     Madison Julius Cawein  B1 λ 0.0  8.0 0.60     0%
     Madison Julius Cawein B1 λ 0.25  8.0 0.40     0%
     Madison Julius Cawein  B1 λ 0.5  8.0 0.27     0%
     Madison Julius Cawein B1 λ 0.75  8.0 0.00     0%
     Madison Julius Cawein  B1 λ 1.0  8.0 0.00     0%
     Madison Julius Cawein   M λ 0.0  8.0 0.93     0%
     Madison Julius Cawein  M λ 0.25  8.0 0.73     0%
     Madison Julius Cawein   M λ 0.5  8.0 0.20     0%
     Madison Julius Cawein  M λ 0.75  8.0 0.00     0%
     Madison Julius Cawein   M λ 1.0  8.0 0.00     0%
Algernon Charles Swinburne      real 10.0 0.95       
Algernon Charles Swinburne  B1 λ 0.0  9.0 0.53     0%
Algernon Charles Swinburne B1 λ 0.25  9.0 0.13     0%
Algernon Charles Swinburne  

In [9]:
# one of each (M, λ 0.25, or λ 1 for Longfellow), next to the start of a real one
for poet in showcase.index:
    lam = 0.25 if showcase.loc[poet, "scheme"] != "none" else 1.0
    gen = [r["poem"] for r in final if r["poet"] == poet and r["system"] == "M" and r["lam"] == lam][0]
    real = inspiring[inspiring["poet"] == poet].iloc[0]
    print("=" * 60)
    print(poet.upper())
    print("-- real:", real["title"])
    print("\n".join(real["text"].strip().split("\n")[:8]))
    print("-- generated:")
    print("\n".join(gen.split("\n")[:8]))
    print()

HENRY WADSWORTH LONGFELLOW
-- real: Autumn
With what a glory comes and goes the year!
The buds of spring, those beautiful harbingers
Of sunny skies and cloudless times, enjoy
Life's newness, and earth's garniture spread out;
And when the silver habit of the clouds
Comes down upon the autumn sun, and with
A sober gladness the old year takes up
His bright inheritance of golden fruits,
-- generated:
And I will yet more nectar deliver,
To my own soul and ours.
III. The Old Priest's Tale of Redemption
There was an old man, that alone
The fair Lyre could call to him being,
In a richly-moulded chamber,
Where the windows showed along
An avenue vast with flowers

MADISON JULIUS CAWEIN
-- real: The Festival Of The Aisne
Imperial Madness, will of hand,
Builds vast an altar here, and rears
Before the world, on godly land,
A Moloch form of blood and tears.
And far as eye can see, behold,
Priests plunge into its brazen arms
Men, that its iron maw of mold
Mangles, returning horrible forms.
-- generat